In [2]:
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import pandas as pd


In [40]:
df = pd.read_excel('app17.xlsx')
df.set_index('etapa', inplace=True)
df_trabalho = df[df.columns[0:2]]
df_trabalho
# minutos necessários

,incandescente,fluorescente
etapa,,
Etapa 1,18,36
Etapa 2,27,45
Etapa 3,9,9


In [53]:
df['disponivel']['Etapa 1']

np.int64(15000)

In [22]:
lampadas = df.columns.tolist()[0:2]
etapas = df.index.tolist()

demanda = {
    lampadas[0]:45000,
    lampadas[1]:22500
}
custo_fab = {
    lampadas[0]:8.25,
    lampadas[1]:12.75
}
custo_compra = {
    lampadas[0]:10.05,
    lampadas[1]:14.25
}



In [23]:
etapas

['Etapa 1', 'Etapa 2', 'Etapa 3']

In [49]:
df_trabalho = df_trabalho.T

In [50]:
df_trabalho

etapa,Etapa 1,Etapa 2,Etapa 3
incandescente,18,27,9
fluorescente,36,45,9


In [70]:
model = pyo.ConcreteModel()

model.lampadas = pyo.Set(initialize = lampadas)
model.etapas = pyo.Set(initialize = etapas)

# x quantidade produzida  e y quantidade comprada
model.x = pyo.Var(model.lampadas, bounds=(0, None), domain=pyo.NonNegativeIntegers)
model.y = pyo.Var(model.lampadas, bounds=(0, None), domain=pyo.NonNegativeIntegers)


def obj(model):
    quantidade_produzida = sum(model.x[l] * custo_fab[l]  for l in model.lampadas)
    quantidade_comprada = sum(model.y[l] * custo_compra[l] for l in model.lampadas)
    return quantidade_produzida + quantidade_comprada
model.objetivo = pyo.Objective(rule=obj, sense=pyo.minimize)

def restricoes_tempo(model, e):
    return sum(df_trabalho[e][l]*model.x[l] for l in model.lampadas) <= df['disponivel'][e]
model.restricoes_tempo = pyo.Constraint(model.etapas, rule=restricoes_tempo)


def restricoes_demanda(model, l):
    return model.x[l] + model.y[l] >= demanda[l]
model.restricao_d = pyo.Constraint(model.lampadas, rule=restricoes_demanda)

In [71]:
model.pprint()

2 Set Declarations
    etapas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'Etapa 1', 'Etapa 2', 'Etapa 3'}
    lampadas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'incandescente', 'fluorescente'}

2 Var Declarations
    x : Size=2, Index=lampadas
        Key           : Lower : Value : Upper : Fixed : Stale : Domain
         fluorescente :     0 :  None :  None : False :  True : NonNegativeIntegers
        incandescente :     0 :  None :  None : False :  True : NonNegativeIntegers
    y : Size=2, Index=lampadas
        Key           : Lower : Value : Upper : Fixed : Stale : Domain
         fluorescente :     0 :  None :  None : False :  True : NonNegativeIntegers
        incandescente :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    objetivo : Size=1, Index=None, Active=True
    

In [72]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmp59dv6u1n.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmplduejt7p.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmplduejt7p.pyomo.lp
Objective sense      : Minimize
Variables            :       4  [General Integer: 4]
Objective nonzeros   :       4
Linear constraints   :       5  [Less: 3,  Greater: 2]
  Nonzeros           :      10
  RHS nonzeros       :       5

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 8.

In [73]:
for m in model.x:
    print(f"Fabricar {pyo.value(model.x[m])} de {m}")
for m in model.y:
    print(f"Comprar {pyo.value(model.y[m])} de {m}")

Fabricar 833.0 de incandescente
Fabricar -0.0 de fluorescente
Comprar 44167.0 de incandescente
Comprar 22500.0 de fluorescente
